ШАГ 7: КЛАССИФИКАЦИЯ (SI > 8); ПОИСК ДАЛЬНЕЙШИХ КАНДИДАТОВ

Целью данного этапа является построение бинарного классификатора для выявления химических соединений с индексом селективности выше заданного порога. Решение данной задачи позволяет отбирать объекты для дальнейшего анализа.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('cleaned_data.csv')

targets_to_exclude = ['IC50, mM', 'CC50, mM', 'SI', 'pIC50', 'pCC50', 'log_SI',
                      'IC50_above_med', 'CC50_above_med', 'SI_above_med', 'SI_above_8']
X = df.drop(columns=targets_to_exclude)
y = df['SI_above_8']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")
print(f"Доля класса 1 (Train): {y_train.mean():.2%}")
print(f"Доля класса 1 (Test): {y_test.mean():.2%}\n")

pipelines_clf = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
    ]),
    'SVC': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(class_weight='balanced', probability=True, random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(class_weight='balanced', random_state=42))
    ])
}

param_grids_clf = {
    'LogisticRegression': {
        'model__C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'model__penalty': ['l2']
    },
    'SVC': {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear']
    },
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5]
    }
}

results = []

for name in pipelines_clf:
    grid = GridSearchCV(pipelines_clf[name], param_grids_clf[name], cv=5,
                        scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    if hasattr(best_model['model'], "predict_proba"):
        y_score = best_model.predict_proba(X_test)[:, 1]
    else:
        y_score = best_model.decision_function(X_test)
    pr_auc = average_precision_score(y_test, y_score)

    results.append({
        'модель': name,
        'лучшие параметры': str(grid.best_params_),
        'F1 Score': f1_score(y_test, y_pred),
        'PR-AUC': pr_auc,
        'Recall': recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values(by='F1 Score', ascending=False)
print("результаты классификации:")
print(results_df.to_string(index=False))

Данные загружены
Train: X=(800, 139), y=(800,)
Test:  X=(201, 139), y=(201,)
Доля класса 1 (Train): 35.62%
Доля класса 1 (Test): 35.82%

Обучение LogisticRegression...
Обучение SVC...
Обучение RandomForest...

Результаты классификации (SI_above_8):
            Модель                                                                    Лучшие параметры  F1 Score   PR-AUC   Recall  Precision  Accuracy
               SVC                                            {'model__C': 10, 'model__kernel': 'rbf'}  0.580645 0.629387 0.625000   0.542169  0.676617
LogisticRegression                                           {'model__C': 0.1, 'model__penalty': 'l2'}  0.550000 0.511720 0.611111   0.500000  0.641791
      RandomForest {'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 200}  0.542636 0.694264 0.486111   0.614035  0.706468


ВЫВОДЫ: 

Среди рассмотренных алгоритмов наилучшее значение целевой F1-меры (около 0.581) показал метод опорных векторов с RBF-ядром, обеспечивший высокую полноту (Recall = 0.625), что свидетельствует об успешном обнаружении значительной доли объектов положительного класса. Случайный лес, в свою очередь, продемонстрировал максимальную площадь под Precision-Recall кривой (PR-AUC ≈ 0.694) и наибольшую точность (Precision = 0.614), однако при более низкой полноте (Recall = 0.486). Это указывает на консервативность модели, склонной относить объекты к положительному классу лишь при высокой уверенности. Логистическая регрессия показала более слабый результат (F1-мера около 0.550), что подтверждает наличие в данных нелинейных зависимостей, успешнее улавливаемых древовидными и ядерными методами.

Для дальнейшего повышения качества предсказаний я предлагаю ряд направлений. Во-первых, для случайного леса, имеющего высокий показатель PR-AUC, целесообразно снизить порог принятия решения (например, до 0.35–0.40 вместо стандартного 0.5), что потенциально увеличит Recall. Во-вторых, применение более мощных ансамблевых методов — градиентного бустинга (XGBoost, LightGBM, CatBoost) — может повысить общую точность. В-третьих, необходима дальнейшая работа с признаками: дополнительный отбор информативных дескрипторов, удаление сильно коррелирующих переменных или использование метода главных компонент (PCA) для снижения размерности помогут уменьшить шум и ограничить переобучение. Наконец, сбор дополнительных данных позволил бы моделям выявлять более сложные закономерности и улучшить обобщающую способность.

